# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Runs offline on the bundled slice.

## 1. My lane as an ML task (type)

**Ranking / scoring.** The output is not a class label per page but an *order*: which pages an editor
should open first. Concretely I train a classifier (is this page about to decline?) and use its score
only as a **ranking key**, because the editor's budget is the thing being allocated.

**The frame in one paragraph.** One row is one already-visible page. Features describe the page's
trailing search and engagement behaviour. The target is whether that page declines in the *next* 30-day
window. The deliverable is a precision@K number on whole held-out clients, compared against a
transparent rule.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


## 2. Target or proxy

**The label here is defined from the data's own two windows**, not invented: a page counts as declining
when next-30 impressions fall below 80% of prior-30 impressions. That is exactly the slice's
`trend_direction == 'down'` rule, so the label is auditable — and it is a **forward** label
(features come from the prior window, the outcome from the later one).

In [2]:
parity = (pool["declined_next_30d"].astype(bool)
          == (pool["trend_direction"].astype(str).str.lower() == "down")).mean()
print(f"label == (trend_direction == 'down') for {100*parity:.1f}% of pool rows")
print("label definition: declined_next_30d = impressions_last_30d < 0.8 * impressions_prev_30d")
print("this is a FORWARD label: prior-30 window in, later-30 window out.")

label == (trend_direction == 'down') for 100.0% of pool rows
label definition: declined_next_30d = impressions_last_30d < 0.8 * impressions_prev_30d
this is a FORWARD label: prior-30 window in, later-30 window out.


## 3. Success metric

**One defendable number: precision@K at the top of the ranked queue — primary K = 50** (about one
editor-week). It must always be printed next to the base rate and ROC-AUC. Accuracy is never a headline:
with tens of thousands of pages an editor will never open, accuracy rewards the majority class.

In [3]:
KS = (10, 20, 50, 100)
print(f"base rate a random queue gives you at any K: {y.mean():.3f}")
print(f"a useful queue clears that by a margin at K=10..100; K=50 is the primary number.")
print(f"pool size {len(pool):,} pages, {len(pool)/50:.0f} filled K=50 slots; 'good' means lift over {y.mean():.3f}")

base rate a random queue gives you at any K: 0.616
a useful queue clears that by a margin at K=10..100; K=50 is the primary number.
pool size 18,010 pages, 360 filled K=50 slots; 'good' means lift over 0.616


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page)**, in the prior-30 window, for a page that is already visible.

In [4]:
cols = ["content_id", "client_id", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev",
        "days_since_last_update", "content_age_days", "declined_next_30d"]
print(f"rows: {len(pool):,} | unique content_id: {pool['content_id'].nunique():,} | "
      f"clients: {pool['client_id'].nunique()}")
print(pool[cols].head(5).to_string(index=False))

rows: 18,010 | unique content_id: 18,010 | clients: 30
          content_id         client_id  impressions_prev_30d  clicks_prev_30d  ctr_prev  days_since_last_update  content_age_days  declined_next_30d
content_304f48230142 client_f369cb89fc                   987               13  1.317123                      20               187                  1
content_a1fb4e703a9e client_4e07408562                  5915                1  0.016906                      25               445                  1
content_9aa793d4d895 client_7f2253d7e2                  6089                3  0.049269                      20               141                  1
content_331d6c4de07b client_19581e27de                  4206               17  0.404184                      22               463                  0
content_d99b7a2d90ca client_3fdba35f04                  6452                2  0.030998                      14               263                  1


## 5. Why ML beats a fixed rule here

**The hypothesis (and the honest caveat).** A fixed rule is a *prior*: "low CTR at high exposure is
risky". A learned model could in principle combine many weak trailing signals that no human would
bother to threshold. But this is a *testable* claim, not an assumption — and on a single anonymized
snapshot the plausible outcome is that the model only ties the rule. That comparison is the whole
capstone, so it is measured on identical held-out clients in `w05_model.ipynb` and audited in
`w06_validation_audit.ipynb`; this cell is only the hypothesis check.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42).split(X, y, groups))
lr = LogisticRegression(max_iter=5000, random_state=42).fit(X.iloc[tr], y[tr])
s_lr = lr.predict_proba(X.iloc[te])[:, 1]
print(f"quick pre-check on {len(te):,} held-out pages / {pd.Series(groups[te]).nunique()} clients "
      f"(base {y[te].mean():.3f}):")
print("  rule  P@50", pk(rule[te], y[te])["50"], "| LR P@50", pk(s_lr, y[te])["50"])
print("  => the model is NOT guaranteed to win; the honest test is in w05/w06.")

quick pre-check on 3,571 held-out pages / 9 clients (base 0.567):
  rule  P@50 0.76 | LR P@50 0.72
  => the model is NOT guaranteed to win; the honest test is in w05/w06.


C:\Users\Ananya\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Self-check

- [x] Task type named (ranking/scoring), not just "classification"
- [x] Target defined, with its data provenance and window direction
- [x] One metric chosen (precision@K, K=50) with the base rate always shown
- [x] Unit of analysis shown as a real dataframe
- [x] The "why ML" claim is framed as a test, not a promise
- [x] Committed under `work/notebooks/`